<a href="https://colab.research.google.com/github/sastagordonramsay/repo5-task-fmri-glm-working-memory/blob/main/notebooks/01_data_expansion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 01_data_expansion.ipynb

!pip install openneuro-py nilearn pandas matplotlib seaborn pybids -q

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create folders
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("figures", exist_ok=True)

print("Project folders ready.")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.1/229.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 12.6 MB/s eta 0:00:00
Project folders ready.


In [11]:
!openneuro-py download --dataset=ds003849 --target-dir=data/raw > download_log.txt 2>&1
print("Download complete. Logs saved to download_log.txt")

Download complete. Logs saved to download_log.txt


In [3]:
# Inspect dataset structure

for root, dirs, files in os.walk("data/raw"):
    level = root.replace("data/raw", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:5]:
        print(" " * 2 * (level + 1) + f)

raw/
  CHANGES
  README.md
  .gitattributes
  dataset_description.json
  sub-BPD0131A/
    ses-01/
      sub-BPD0131A_ses-01_scans.tsv
      func/
        sub-BPD0131A_ses-01_task-nback_run-01_bold.nii.gz
        sub-BPD0131A_ses-01_task-rest_run-01_events.tsv
        sub-BPD0131A_ses-01_task-nback_run-01_events.tsv
        sub-BPD0131A_ses-01_task-nback_run-01_bold.json
        sub-BPD0131A_ses-01_task-rest_run-01_bold.json
      anat/
        sub-BPD0131A_ses-01_run-01_T1w.json
        sub-BPD0131A_ses-01_run-01_T2w.json
        sub-BPD0131A_ses-01_run-01_T2w.nii.gz
        sub-BPD0131A_ses-01_run-01_T1w.nii.gz
  sub-BPD0103B/
    ses-01/
      sub-BPD0103B_ses-01_scans.tsv
      func/
        sub-BPD0103B_ses-01_task-rest_run-01_bold.json
        sub-BPD0103B_ses-01_task-nback_run-01_bold.json
        sub-BPD0103B_ses-01_task-nback_run-01_events.tsv
        sub-BPD0103B_ses-01_task-rest_run-01_events.tsv
        sub-BPD0103B_ses-01_task-rest_run-01_bold.nii.gz
      anat/
        su

In [4]:
# Load participants file if present

participants_path_tsv = "data/raw/ds003849/participants.tsv"
participants_path_csv = "data/raw/ds003849/participants.csv"

if os.path.exists(participants_path_tsv):
    participants = pd.read_csv(participants_path_tsv, sep="\t")
elif os.path.exists(participants_path_csv):
    participants = pd.read_csv(participants_path_csv)
else:
    participants = None
    print("No participants file found.")

if participants is not None:
    print("Subjects downloaded:", participants.shape[0])
    display(participants.head())

No participants file found.


In [5]:
# Basic participant audit

if participants is not None:
    print("\nColumns:")
    print(participants.columns.tolist())

    print("\nMissing values:")
    print(participants.isnull().sum())

In [7]:
# Visual summaries (dynamic based on available columns)

if participants is not None:

    numeric_cols = participants.select_dtypes(include=np.number).columns.tolist()
    cat_cols = participants.select_dtypes(exclude=np.number).columns.tolist()

    # Plot first numeric column if available
    if len(numeric_cols) > 0:
        plt.figure(figsize=(8,5))
        sns.histplot(participants[numeric_cols[0]].dropna(), bins=15, kde=True)
        plt.title(f"Distribution of {numeric_cols[0]}")
        plt.show()

    # Plot first categorical column if available
    if len(cat_cols) > 0:
        plt.figure(figsize=(10,5))
        sns.countplot(x=participants[cat_cols[0]])
        plt.xticks(rotation=45)
        plt.title(f"Counts: {cat_cols[0]}")
        plt.show()

In [8]:
# Search for task/event files related to working memory

event_files = []

for root, dirs, files in os.walk("data/raw"):
    for f in files:
        if "events.tsv" in f.lower():
            event_files.append(os.path.join(root, f))

print("Number of event files found:", len(event_files))
event_files[:10]

Number of event files found: 184


['data/raw/sub-BPD0131A/ses-01/func/sub-BPD0131A_ses-01_task-rest_run-01_events.tsv',
 'data/raw/sub-BPD0131A/ses-01/func/sub-BPD0131A_ses-01_task-nback_run-01_events.tsv',
 'data/raw/sub-BPD0103B/ses-01/func/sub-BPD0103B_ses-01_task-nback_run-01_events.tsv',
 'data/raw/sub-BPD0103B/ses-01/func/sub-BPD0103B_ses-01_task-rest_run-01_events.tsv',
 'data/raw/sub-BPD0123B/ses-01/func/sub-BPD0123B_ses-01_task-nback_run-01_events.tsv',
 'data/raw/sub-BPD0123B/ses-01/func/sub-BPD0123B_ses-01_task-rest_run-01_events.tsv',
 'data/raw/sub-BPD0099A/ses-01/func/sub-BPD0099A_ses-01_task-nback_run-01_events.tsv',
 'data/raw/sub-BPD0099A/ses-01/func/sub-BPD0099A_ses-01_task-rest_run-01_events.tsv',
 'data/raw/sub-BPD0105B/ses-01/func/sub-BPD0105B_ses-01_task-rest_run-01_events.tsv',
 'data/raw/sub-BPD0105B/ses-01/func/sub-BPD0105B_ses-01_task-nback_run-01_events.tsv']

In [9]:
# Preview first events file

if len(event_files) > 0:
    events = pd.read_csv(event_files[0], sep="\t")
    display(events.head())

    events.to_csv("data/processed/sample_events_preview.csv", index=False)

,onset,duration,trial_type,response_time,stim_file,TODO -- fill in rows and add more tab-separated columns if desired


In [10]:
# Save expanded metadata

if participants is not None:
    participants.to_csv(
        "data/processed/participants_expanded.csv",
        index=False
    )

print("Saved processed metadata.")

Saved processed metadata.
